# Smart City-project van Nova Stad

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, lit, udf, pandas_udf, PandasUDFType
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder, Imputer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


ModuleNotFoundError: No module named 'pyspark'

# Data inladen

In [24]:
import os

img_path = "bdd/images/train"
label_path = "bdd/labels/train"

# haal image namen zonder .jpg
image_ids = set([f.replace(".jpg", "") for f in os.listdir(img_path) if f.endswith(".jpg")])

# labels ophalen
labels = [f for f in os.listdir(label_path) if f.endswith(".json")]

# check matches
matched = [f for f in labels if f.replace(".json", "") in image_ids]

print("Aantal images", len(image_ids))
print("Aantal labels", len(labels))
print("Matches", len(matched))

Aantal images 2476
Aantal labels 2476
Matches 2476


### label verwijderen

In dit code gedeelte checken we matching labels en images. Niet gekoppelde items worden verwijderd.

In [13]:
matched_set = set([f.replace(".json", "") for f in matched])

In [14]:
for f in labels:
    base = f.replace(".json", "")
    
    if base not in matched_set:
        os.remove(os.path.join(label_path, f))

print("Alleen matched labels behouden ")

Alleen matched labels behouden 


In [15]:
remaining_labels = [f for f in os.listdir(label_path) if f.endswith(".json")]

print("Overgebleven labels", len(remaining_labels))

Overgebleven labels 2476


### Images verwijderen

In [ ]:
image_files = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
label_ids = set(os.path.splitext(f)[0] for f in os.listdir(label_path) if f.endswith(".json"))

delete_images = [f for f in image_files if os.path.splitext(f)[0] not in label_ids]


In [23]:
for f in delete_images:
    os.remove(os.path.join(img_path, f))

print("Images opgeschoond")

Images opgeschoond


In [18]:
remaining_images = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
remaining_labels = [f for f in os.listdir(label_path) if f.endswith(".json")]

print("Images:", len(remaining_images))
print("Labels:", len(remaining_labels))

Images: 2476
Labels: 2476


## Verdeel images en labels naar val map

In [25]:
import os
import shutil
import random

img_path = "bdd/images/train"
val_img = "bdd/images/val"
label_path = "bdd/labels/train"
val_label = "bdd/labels/val"

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_label, exist_ok=True)

#  Verwijder alle val images
#for f in os.listdir(val_img):
    #if f.endswith(".jpg"):
        #os.remove(os.path.join(val_img, f))

# Verwijder alle val labels
#for f in os.listdir(val_label):
    #if f.endswith(".json"):
        #os.remove(os.path.join(val_label, f))

# Kies alleen train images die ook een label hebben
image_files = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
label_ids = {os.path.splitext(f)[0] for f in os.listdir(label_path) if f.endswith(".json")}


matched_images = [f for f in image_files if os.path.splitext(f)[0] in label_ids]

print("Beschikbare matched train images", len(matched_images))

# Selecteer 500 images voor val
#random.seed(42)
#val_selection = random.sample(matched_images, 500)

# Verplaats image + label naar val
#for img_file in val_selection:
    #base = os.path.splitext(img_file)[0]
    #label_file = base + ".json"

    #shutil.move(os.path.join(img_path, img_file), os.path.join(val_img, img_file))
    #shutil.move(os.path.join(label_path, label_file), os.path.join(val_label, label_file))

#print("Klaar")

Beschikbare matched train images 2476


# test

In [26]:
def count_files(path, ext):
    return len([f for f in os.listdir(path) if f.endswith(ext)])

print("Train images:", count_files(img_path, ".jpg"))
print("Train labels:", count_files(label_path, ".json"))
print("Val images:", count_files(val_img, ".jpg"))
print("Val labels:", count_files(val_label, ".json"))

Train images: 2476
Train labels: 2476
Val images: 500
Val labels: 500


## Data pipeline

In [0]:
df_traffic = spark.read.csv("/FileStore/tables/metro_interstate_traffic_volume_train.csv", header=True, inferSchema=True)

In [0]:
df = spark.read.csv(
    "/Volumes/main/default/datasets/metro_interstate_traffic_volume_train.csv",
    header=True,
    inferSchema=True
)

display(df.head(5))

In [0]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, to_timestamp, year, month, dayofmonth, hour, dayofweek
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# Pad
train_path = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
test_path  = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"
# functie voor schoonmaken
def clean_df(df):

    # 1. duplicaten verwijderen
    df = df.dropDuplicates()

    # 2. datum omzetten
    df = df.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))

    # 3. string kolommen opschonen
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(c, regexp_replace(col(c), r"\s+", " "))

    # 4. numerieke kolommen bepalen
    numeric_types = ["int", "bigint", "double", "float", "long", "short", "decimal"]
    numeric_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() in numeric_types]

    # 5. missende numerieke waardes vullen (mediaan)
    for c in numeric_cols:
        median_value = df.approxQuantile(c, [0.5], 0.01)[0]
        df = df.fillna({c: median_value})

    # 6. missende string waardes vullen
    for c in string_cols:
        df = df.fillna({c: "Unknown"})

    # 7. datum features maken
    df = (
        df
        .withColumn("year", year(col("date_time")))
        .withColumn("month", month(col("date_time")))
        .withColumn("day", dayofmonth(col("date_time")))
        .withColumn("hour", hour(col("date_time")))
        .withColumn("dayofweek", dayofweek(col("date_time")))
    )

    return df


# Alles in 1 lijn uitvoeren (dan wordt neiwue df een keer opgeslagen)
df_train = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(train_path)
)

df_test = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(test_path)
)

# Resultaat bekijken
display(df_train.limit(5))
display(df_test.limit(5))